# 01 — EDA of the DocVQA subset
Run after Phase 1. Set `DATA_DIR` (local: `../data/processed`, Colab: the Drive path).

In [ ]:
import json, collections
from pathlib import Path
import matplotlib.pyplot as plt

DATA_DIR = Path("../data/processed")   # Colab: Path("/content/data/processed")
stats = json.load(open(DATA_DIR / "stats.json"))
rows = {s: [json.loads(l) for l in open(DATA_DIR / f"{s}.jsonl")] for s in ("train", "val", "test")}
print("questions:", stats["counts"], "| documents:", stats["documents"])
print("image width:", stats["images"]["width"]); print("image height:", stats["images"]["height"])

In [ ]:
allr = [r for v in rows.values() for r in v]
words = [len(r["answers"][0].split()) for r in allr]
chars = [len(r["answers"][0]) for r in allr]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(words, bins=range(1, max(words) + 2)); ax[0].set_title("Answer length (words)"); ax[0].set_xlabel("words")
ax[1].hist(chars, bins=40); ax[1].set_title("Answer length (characters)"); ax[1].set_xlabel("characters")
plt.tight_layout(); plt.show()

In [ ]:
imgs = {r["image"]: (r["width"], r["height"]) for r in allr}
w, h = zip(*imgs.values())
plt.figure(figsize=(6, 5)); plt.scatter(w, h, s=6, alpha=.5)
plt.xlabel("width (px)"); plt.ylabel("height (px)"); plt.title(f"Image resolutions ({len(imgs)} unique pages)")
mp = [a * b / 1e6 for a, b in imgs.values()]
print(f"megapixels: median {sorted(mp)[len(mp)//2]:.2f}; max_pixels=1280*28*28 = {1280*28*28/1e6:.2f} MP -> most pages are DOWNscaled")
plt.show()

In [ ]:
types = collections.Counter(t for r in allr for t in r["question_types"])
plt.figure(figsize=(8, 4)); plt.bar(list(types), list(types.values())); plt.xticks(rotation=30, ha="right")
plt.title("question_types (multi-label)"); plt.tight_layout(); plt.show()
for s, v in rows.items():
    print(s, collections.Counter(t for r in v for t in r["question_types"]).most_common(4))

In [ ]:
from PIL import Image
r = rows["train"][0]
print(r["question"], "->", r["answers"])
Image.open(DATA_DIR / r["image"]).resize((500, int(500 * r["height"] / r["width"])))